# Kaizen Video: Upload Pictures, Get a Narrated Video

1. Run each cell in order.
2. Upload **10 pictures** (`1.png` to `10.png`) and **reference_30s.mp3** when asked.
3. The narration is made in your reference voice, one clip per slide.
4. The video has plain cuts between slides. No zoom, no crossfades.
5. You get a download and a notification when it's done.


In [ ]:
# Step 1: install tools (about 1-2 minutes the first time)
!pip install -q f5-tts pillow
import torch
print("GPU on:", torch.cuda.is_available())


In [ ]:
# Step 2: upload pictures and the reference voice
import os, shutil
from google.colab import files

os.makedirs("work/images", exist_ok=True)
os.makedirs("work/narr", exist_ok=True)

print("Choose your 10 pictures (1.png ... 10.png) AND reference_30s.mp3")
up = files.upload()
for name in up:
    low = name.lower()
    if low.endswith(".mp3"):
        shutil.move(name, "work/reference.mp3")
    elif low.endswith((".png", ".jpg", ".jpeg")):
        shutil.move(name, "work/images/" + name)
    else:
        os.remove(name)

missing = [n for n in range(1, 11) if not any(
    os.path.exists(f"work/images/{n}{e}") for e in (".png", ".jpg", ".jpeg"))]
print("Reference voice:", os.path.exists("work/reference.mp3"))
print("Missing pictures:", missing or "none")


In [ ]:
# Step 3: the script (one line per slide, in order). Edit here if you want changes.
SLIDES = [
    "Okay so, let's talk about Kaizen. It's a simple idea, with a big impact. And no tension, you can start today.",
    "Kaizen is a continuous improvement philosophy. Basically, everyone makes small changes, every day. And trust me, those small changes add up fast. Many small improvements lead to extraordinary results.",
    "So what's the purpose? It helps you cut waste. It improves quality. It boosts productivity. It makes work safer. It reduces cost. And it gets everyone involved. That's the vibe.",
    "There are three types. First, Daily Kaizen. Small, quick, and low-cost. Next, Focused Kaizen. It targets one specific problem. And last, Breakthrough Kaizen. That's the big one, with major changes and bigger impact.",
    "The process is PDCA. Plan the change. Do it on a small scale. Check the results. And then, Act, to make it stick. Simple... and it works.",
    "Here are the key tools. Gemba Walk. Five Why. Ishikawa. And more. Pick the right tool for the problem. And you'll find the waste, fast.",
    "And the benefits are real. Safer work. Better quality. More output. Faster flow. Lower cost. And happier customers. No tension... the numbers show it.",
    "Now, the eight wastes. We call them TIMWOODS. Find these in your daily work. And you've found your next improvement.",
    "What makes Kaizen work? Leadership support. Involving everyone. Small steps. Measuring results. Standardizing. And repeating, every day. Trust me... that's how it becomes a way of life.",
    "Remember, Kaizen is everyone's responsibility. A better workplace is built by many small improvements, every day. Thanks for watching!",
]
print("Slides in script:", len(SLIDES))


In [ ]:
# Step 4: make the narration for each slide in your reference voice (takes a few minutes)
from f5_tts.api import F5TTS

tts = F5TTS(device="cuda" if torch.cuda.is_available() else "cpu")
for i, text in enumerate(SLIDES, start=1):
    tts.infer(ref_file="work/reference.mp3", ref_text="", gen_text=text,
              file_wave=f"work/narr/{i:02d}.wav")
    print("Narration done:", i, "of", len(SLIDES))


In [ ]:
# Step 5: build the video with plain cuts (one picture per slide, timed to its narration)
import subprocess

def dur(path):
    out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                          "-of", "csv=p=0", path], capture_output=True, text=True).stdout
    return float(out.strip())

clips = []
for i in range(1, len(SLIDES) + 1):
    pic = next(p for p in (f"work/images/{i}.png", f"work/images/{i}.jpg", f"work/images/{i}.jpeg")
               if os.path.exists(p))
    clip = f"work/clip_{i:02d}.mp4"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-loop", "1", "-i", pic,
                    "-i", f"work/narr/{i:02d}.wav",
                    "-vf", "scale=1920:1080:force_original_aspect_ratio=decrease,"
                           "pad=1920:1080:(ow-iw)/2:(oh-ih)/2:white,format=yuv420p",
                    "-c:v", "libx264", "-preset", "slow", "-crf", "18", "-pix_fmt", "yuv420p",
                    "-c:a", "aac", "-b:a", "192k", "-shortest", clip], check=True)
    clips.append(os.path.basename(clip))

with open("work/list.txt", "w") as f:
    for c in clips:
        f.write(f"file '{c}'\n")
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0",
                "-i", "list.txt", "-c", "copy", "Kaizen_Final.mp4"], cwd="work", check=True)
print("Video ready: work/Kaizen_Final.mp4")
print("Length (sec):", round(dur("work/Kaizen_Final.mp4"), 1))


In [ ]:
# Step 6: download the video
from google.colab import files
files.download("work/Kaizen_Final.mp4")


In [ ]:
# Step 7: notification when everything is done
from IPython.display import Audio, display, HTML

display(HTML("<h2 style='color:green'>✅ Kaizen video is ready!</h2>"))
display(Audio(url="https://actions.google.com/sounds/v1/alarms/beep_short.ogg", autoplay=True))
